<!--
File: notebooks_v2/01_dataset_creation_v2.ipynb
What does this file do?
    Explains and inspects the Nepal-first V2 dataset creation workflow.
Methods/functions this file contains:
    Notebook cells for loading generated CSV files, checking schema, and summarizing dataset quality.
Date and Day of last modification:
    2026-05-21, Thursday.
-->

# 01 - Nepal Finance Dataset Creation V2

This notebook documents the dataset side of the project. The V2 dataset is Nepal-first, NPR-normalized, multi-currency aware, and includes personal plus shared project expenses.

Main generated folder: `output_v2/`

## Dataset Design

The dataset includes:

- users and financial personas
- income events
- budgets and goals
- personal expenses
- project/shared expenses
- expense splits
- recurring payments
- currency rates
- festival calendar

The aim is not just to predict expenses, but to provide financial assistance: budget risk, unusual spending, category focus, and shared-settlement insight.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks_v2' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'output_v2'
files = sorted(p.name for p in DATA_DIR.glob('*.csv'))
files

['budgets.csv',
 'currency_rates.csv',
 'expense_splits.csv',
 'festival_calendar.csv',
 'goals.csv',
 'group_memberships.csv',
 'groups.csv',
 'income_events.csv',
 'recurring_payments.csv',
 'shared_expenses.csv',
 'users.csv']

In [2]:
frames = {path.stem: pd.read_csv(path, low_memory=False) for path in DATA_DIR.glob('*.csv')}
summary = pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns)}
    for name, df in frames.items()
]).sort_values('table')
summary

,table,rows,columns
0,budgets,5700,4
1,currency_rates,3,3
2,expense_splits,121468,7
3,festival_calendar,4,5
4,goals,300,5
6,group_memberships,621,5
5,groups,120,5
7,income_events,10800,6
8,recurring_payments,39755,6
9,shared_expenses,25237,8


In [5]:
expenses = frames['shared_expenses'].copy()
print('Users:', len(frames['users']))
print('Shared expenses:', len(expenses))
print('Expense splits:', len(frames['expense_splits']))

# Robust schema-aware summary
date_col = next((c for c in ['date', 'created_at', 'expense_date', 'timestamp'] if c in expenses.columns), None)
currency_col = next((c for c in ['currency_code', 'currency', 'currency_id'] if c in expenses.columns), None)

if date_col is not None:
    expenses[date_col] = pd.to_datetime(expenses[date_col], errors='coerce')
    print('Date range:', expenses[date_col].min(), 'to', expenses[date_col].max())
else:
    print('Date column not present in shared_expenses (expected in this V2 export).')

if currency_col is not None:
    print('Currencies:', expenses[currency_col].value_counts().to_dict())
else:
    print('Currency column not present in shared_expenses (expected in this V2 export).')

print('Shared expense columns:', list(expenses.columns))


Users: 300
Shared expenses: 25237
Expense splits: 121468
Date column not present in shared_expenses (expected in this V2 export).
Currency column not present in shared_expenses (expected in this V2 export).
Shared expense columns: ['shared_expense_id', 'expense_id', 'group_id', 'paid_by_user_id', 'split_type', 'participant_count', 'settlement_status', 'days_to_settle']


In [7]:
if {'category', 'amount_npr'}.issubset(expenses.columns):
    category_summary = expenses.groupby('category').agg(
        rows=('expense_id', 'count'),
        total_npr=('amount_npr', 'sum'),
        median_npr=('amount_npr', 'median'),
    ).sort_values('total_npr', ascending=False)
    display(category_summary.head(12))
else:
    print("'shared_expenses' does not contain category/amount_npr columns.")
    print("Showing fallback from recurring_payments by category.")
    rp = frames.get('recurring_payments')
    if rp is not None and {'category', 'expected_amount_npr'}.issubset(rp.columns):
        fallback_summary = rp.groupby('category').agg(
            rows=('recurring_id', 'count'),
            total_npr=('expected_amount_npr', 'sum'),
            median_npr=('expected_amount_npr', 'median'),
        ).sort_values('total_npr', ascending=False)
        display(fallback_summary.head(12))
    else:
        print('No suitable fallback table found for category amount summary.')


'shared_expenses' does not contain category/amount_npr columns.
Showing fallback from recurring_payments by category.


,rows,total_npr,median_npr
category,,,
rent,4879,1.258258e+08,22077.150
family_support,6104,1.224172e+08,16863.290
loan_emi,3409,6.332250e+07,15910.680
savings_investment,3456,3.373241e+07,7286.045
utilities,7756,3.085431e+07,3416.115
internet_mobile,8489,1.870541e+07,1867.970
subscriptions,5662,1.448325e+07,2188.365


In [8]:
users = frames['users']
users['financial_persona'].value_counts(normalize=True).mul(100).round(2)

financial_persona
salaried_balanced              18.67
student_low_budget             16.00
salaried_family_support        14.33
high_social_spender            11.33
freelancer_irregular_income    11.00
rent_pressure_user              8.33
business_owner_cash_heavy       8.00
festival_spike_user             6.33
debt_repayment_user             6.00
Name: proportion, dtype: float64